In [44]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("qoute_dataset.csv")

In [45]:
df = pd.read_csv("qoute_dataset.csv")

In [46]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [47]:
df['quote'][2]

'“There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”'

In [48]:
df.shape

(3038, 2)

In [49]:
quotes = df['quote']
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [50]:
quotes = quotes.str.lower()

In [51]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [52]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [53]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [54]:
vocab_size = 8978

tokinizer = Tokenizer(num_words=vocab_size)
tokinizer.fit_on_texts(quotes)

In [55]:
word_index = tokinizer.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [56]:
sequence = tokinizer.texts_to_sequences(quotes)

In [57]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [58]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [59]:
# for predicting the next word


X = []
y = []

for seq in sequence:
    for i in range(1,len(seq)):
        input_seq = seq[:i]
        output_seq = seq[i]
        X.append(input_seq)
        y.append(output_seq)

In [ ]:
X[1:5]

[[713, 62], [713, 62, 29], [713, 62, 29, 19], [713, 62, 29, 19, 16]]

In [ ]:
y[1:5]

[29, 19, 16, 946]

In [ ]:
len(X)

85270

In [ ]:
len(y)

85270

In [ ]:
# apply padding for doing the same size of x

In [60]:
max_len = max(len(x) for x in X)
print(max_len)

745


In [61]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
X_padded = pad_sequences(X,maxlen = max_len,padding = 'pre')

In [ ]:
X_padded

array([[   0,    0,    0, ...,    0,    0,  713],
       [   0,    0,    0, ...,    0,  713,   62],
       [   0,    0,    0, ...,  713,   62,   29],
       ...,
       [   0,    0,    0, ...,    9,   19, 1125],
       [   0,    0,    0, ...,   19, 1125,    3],
       [   0,    0,    0, ..., 1125,    3,  169]], dtype=int32)

In [62]:
y = np.array(y)

In [63]:
X_padded.shape

(85270, 745)

In [64]:
y.shape

(85270,)

In [65]:
from tensorflow.keras.utils import to_categorical
y_one_hot = to_categorical(y, num_classes=vocab_size)

In [66]:
y_one_hot

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]])

In [67]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,SimpleRNN,LSTM, Dense

In [ ]:
embedding_dim = 50
rnn_units = 128

In [ ]:
rnn_model = Sequential()

rnn_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
rnn_model.add(SimpleRNN(units=rnn_units))
rnn_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
rnn_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [ ]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
epochs = 10
batch_size = 128

In [ ]:
history_rnn = rnn_model.fit(
    X_padded,y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 46s 70ms/step - accuracy: 0.0442 - loss: 6.7053 - val_accuracy: 0.0597 - val_loss: 6.5550
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0713 - loss: 6.1661 - val_accuracy: 0.0840 - val_loss: 6.4063
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.0972 - loss: 5.8233 - val_accuracy: 0.0993 - val_loss: 6.2915
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1140 - loss: 5.5320 - val_accuracy: 0.1036 - val_loss: 6.3150
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1264 - loss: 5.2778 - val_accuracy: 0.1028 - val_loss: 6.3738
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1386 - loss: 5.0419 - val_accuracy: 0.1088 - val_loss: 6.4032
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 63ms/step - accuracy: 0.1524 - loss: 4.8170 - val_accuracy: 0.1095 - val_loss: 6.4633
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 38s 64ms/step - accuracy: 0.1697 - loss: 4.6040 - 

In [40]:
print("hi")

hi


In [68]:
lstm_model = Sequential()
lstm_model.add(
    Embedding(input_dim=vocab_size, output_dim=embedding_dim, input_length=max_len)
)
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size, activation='softmax'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [69]:
lstm_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

In [70]:
lstm_model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [71]:
epochs = 100
batch_size = 128

history_lstm = lstm_model.fit(
    X_padded,y_one_hot,
    epochs = epochs,
    batch_size = batch_size,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 41s 56ms/step - accuracy: 0.0386 - loss: 6.7392 - val_accuracy: 0.0434 - val_loss: 6.6671
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.0604 - loss: 6.2992 - val_accuracy: 0.0697 - val_loss: 6.5188
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.0849 - loss: 6.0137 - val_accuracy: 0.0899 - val_loss: 6.4361
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1004 - loss: 5.7851 - val_accuracy: 0.0983 - val_loss: 6.3822
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 33s 54ms/step - accuracy: 0.1147 - loss: 5.5855 - val_accuracy: 0.1016 - val_loss: 6.3789
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1237 - loss: 5.4094 - val_accuracy: 0.1051 - val_loss: 6.3906
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1323 - loss: 5.2489 - val_accuracy: 0.1060 - val_loss: 6.4246
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1400 - loss: 5

In [72]:
lstm_model.save("lstm_model.h5")

In [74]:
index_to_word = {}
for word,index in word_index.items():
  index_to_word[index] = word

In [75]:
index_to_word

{1: 'the',
 2: 'you',
 3: 'to',
 4: 'and',
 5: 'a',
 6: 'i',
 7: 'is',
 8: 'of',
 9: 'that',
 10: 'it',
 11: 'in',
 12: 'be',
 13: 'not',
 14: 'are',
 15: 'your',
 16: 'have',
 17: 'for',
 18: 'but',
 19: 'we',
 20: 'if',
 21: 'what',
 22: 'with',
 23: 'all',
 24: 'love',
 25: 'can',
 26: 'my',
 27: 'when',
 28: 'will',
 29: 'as',
 30: 'who',
 31: 'do',
 32: 'or',
 33: 'me',
 34: 'he',
 35: 'they',
 36: 'life',
 37: 'one',
 38: 'was',
 39: 'like',
 40: 'there',
 41: 'people',
 42: 'on',
 43: 'its',
 44: 'at',
 45: 'so',
 46: 'never',
 47: 'no',
 48: 'them',
 49: 'dont',
 50: 'know',
 51: 'just',
 52: 'more',
 53: 'only',
 54: 'than',
 55: 'because',
 56: 'this',
 57: 'want',
 58: 'up',
 59: 'how',
 60: 'his',
 61: 'things',
 62: 'world',
 63: 'by',
 64: 'think',
 65: 'make',
 66: 'about',
 67: 'time',
 68: 'from',
 69: 'always',
 70: 'our',
 71: 'an',
 72: 'out',
 73: 'us',
 74: 'good',
 75: 'said',
 76: 'she',
 77: 'her',
 78: 'way',
 79: 'go',
 80: 'am',
 81: 'live',
 82: 'has',
 83:

In [76]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [78]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq],maxlen = max_len,padding = 'pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]

In [85]:
# predicting a word

seeed_text = " how  "
next_word = predictor(lstm_model,tokinizer,seeed_text,max_len)

print(next_word)

many


In [ ]:
# now predicting a sentence

In [83]:
def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text

In [84]:
seed = "are you a "
generate_text = generate_text(lstm_model,tokinizer,seed,max_len,10)
print(generate_text)

are you a  kind of library and a painting and mean it is


In [86]:
import pickle
with open("tokenizer.pkl", "wb") as f:
  pickle.dump(tokinizer, f)

In [87]:
with open("max_len.pkl", "wb") as f:
  pickle.dump(max_len, f)